In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datasets
from privacy_estimates.experiments.aml import JobList
from sklearn.metrics import roc_curve, roc_auc_score, auc
from datetime import datetime

In [2]:
def compute_performance(scores_members, scores_non_members):
    mia_performance = {}
    
    member_vals = [val for val in scores_members]
    non_member_vals = [val for val in scores_non_members]
    mia_performance['auc'] = roc_auc_score([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    fpr, tpr, thresholds = roc_curve([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    for target_fpr in (0.01, 0.05, 0.1):
        mia_performance[f'tpr_at_{target_fpr}'] = np.interp(target_fpr, fpr, tpr)
    print(f"AUC: {mia_performance['auc']}, TPR@0.01: {mia_performance['tpr_at_0.01']}, TPR@0.05: {mia_performance['tpr_at_0.05']}, TPR@0.1: {mia_performance['tpr_at_0.1']}")

    # also add the curves
    mia_performance['fpr'] = fpr
    mia_performance['tpr'] = tpr

    return mia_performance

def compute_performance_from_url(url, job_name = None):
    jobs = JobList.from_urls([url])
    if job_name is None:
        job_name = str(datetime.now())
    
    if not os.path.exists(f'./mia_results/{job_name}'):
        test = jobs[0].get_node('estimate_privacy').download_input('scores', f'./mia_results/{job_name}/scores')
        test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', f'./mia_results/{job_name}/challenge_bits')
    scores = datasets.load_from_disk(f'./mia_results/{job_name}/scores')
    bits = datasets.load_from_disk(f'./mia_results/{job_name}/challenge_bits')
    
    membership_scores = np.array([k['score'] for k in scores])
    membership_labels = np.array([k['challenge_bit'] for k in bits])
    members = membership_scores[membership_labels == 1]
    non_members = membership_scores[membership_labels == 0]
    return compute_performance(members, non_members)

In [3]:
# this is all for syntheticcanary_uniformlabel with n_rep = 12
all_urls = {
    'sst2': {
        'synthetic_2gram': 'https://ml.azure.com/runs/quirky_battery_tvw7wcm4wh?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourcegroups/PPML/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47'
    },
    'agnews': {
        'synthetic_2gram': 'https://ml.azure.com/runs/quirky_insect_7scfdk028l?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourcegroups/PPML/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47',
    },
    'sst2-best': {
        'synthetic_2gram': 'https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/gifted_jewel_w50pc8cd89?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47#'
    },
}

In [4]:
DATASET = 'sst2-best'
text_key = 'sentence'
synthetic_job_name = all_urls[DATASET]['synthetic_2gram']

In [5]:
jobs = JobList.from_urls([synthetic_job_name])
synthetic_job = jobs[0]

**Clean up before proceeding.**

Before you run the below, I recommend running './clean_for_interpet.sh' in the notebooks directory. This makes sure nothing remains from the previous run and everything can be downloaded for again for the right jobs. 

## (1) Let's first get the canaries!

In [ ]:
# get the canaries
test = jobs[0].get_node('add_index_to_dataset_2').get_node('append_column_incrementing').download_input('data', 'canaries_synthetic')
canaries = datasets.load_from_disk('canaries_synthetic')
canaries

In [7]:
# for all reference models, get the canary data that was IN

ref_model_to_in_data = {}

for i in range(1, 5):
    if i == 4:
        test = jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict').get_node('filter_in_data').download_output('filtered', f'in_data_{i}')
    else:
         jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict_{i}').get_node('filter_in_data').download_output('filtered', f'in_data_{i}')
    in_data_for_model = datasets.load_from_disk(f'in_data_{i}')
    ref_model_to_in_data[i] = in_data_for_model

In [ ]:
# for all reference models, get the generated synthetic data

ref_model_to_synthetic_data = {}

for i in range(1, 5):
    if i == 4:
        test = jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_{i}')
    else:
        test =  jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict_{i}').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_{i}')
    synthetic_data = datasets.Dataset.from_json(f'synthetic_data_{i}/prep_synthetic_data_path')
    ref_model_to_synthetic_data[i] = synthetic_data

In [ ]:
canaries[i]

In [ ]:
# let's pick a certain canary


idx = 0
canary = canaries[idx][text_key]
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model[text_key]:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)

In [11]:
# actually we want to look at the most vulnerable when it comes to the target model/RMIA scores

test = jobs[0].get_node('estimate_privacy').download_input('scores', './mia_results/scores_synthetic/scores')
scores_synthetic = datasets.load_from_disk('./mia_results/scores_synthetic/scores')

test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', './mia_results/challenge_bits_synthetic/challenge_bits')
challenge_bits_synthetic = datasets.load_from_disk('./mia_results/challenge_bits_synthetic/challenge_bits')

test =  jobs[0].get_node('train_many_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('data_for_model').download_input('in_out_data', 'in_out_data_synthetic')
in_out_data_synthetic = datasets.load_from_disk('in_out_data_synthetic')

In [ ]:
# also got the target model synthetic data
test = jobs[0].get_node('train_many_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_target')
target_synthetic_data = datasets.Dataset.from_json(f'synthetic_data_target/prep_synthetic_data_path')

In [ ]:
membership_scores_synthetic = [k['score'] for k in scores_synthetic]
membership_labels_synthetic = [k['challenge_bit'] for k in challenge_bits_synthetic]

top_n = 5

high_to_low_indices = np.argsort(membership_scores_synthetic)[::-1]

for idx in high_to_low_indices[:top_n]:
    print(f"Top synthetic score: {membership_scores_synthetic[idx]}, challenge bit: {membership_labels_synthetic[idx]}")
    print(in_out_data_synthetic[int(idx)][text_key])
    print('---')

In [ ]:
most_vulnerable_canary = in_out_data_synthetic[int(high_to_low_indices[0])][text_key]
print("most vulnerable canary: ")
print(most_vulnerable_canary)

What do we now want to do? 

- Run through all models
- Train an n-gram model on the corresponding synthetic data
- Get the log likelihood of the canary for that n-gram model
- And also check what is 'extracted' from the sequence
- This enables us to compare IN and OUT for this sequence

In [15]:
import collections
from tqdm import tqdm

def generate_ngrams(text, n):
    """
    Generate n-grams from the input text.
    """
    tokens = text.split()
    ngrams = zip(*[tokens[i:] for i in range(n)])
    return [' '.join(ngram) for ngram in ngrams]

def train_ngram_model(all_text, n, smoothing=1):
    """
    Train an n-gram model from the given text using Laplace smoothing.
    """
    all_ngrams = []
    vocabulary = set()

    for text in tqdm(all_text):
        words = text.split()
        vocabulary.update(words)
        ngrams = generate_ngrams(text, n)
        all_ngrams.extend(ngrams)

    ngram_counts = collections.Counter(all_ngrams)
    total_ngrams = sum(ngram_counts.values()) + smoothing * len(vocabulary) ** n

    # Convert counts to probabilities with smoothing
    ngram_probabilities = {
        ngram: (count + smoothing) / total_ngrams
        for ngram, count in ngram_counts.items()
    }

    return ngram_probabilities, len(vocabulary), total_ngrams

def inference(ngram_model, text, n, vocabulary_size, total_ngrams, smoothing=1):
    """
    Compute the loss of the n-gram model on a given piece of text.
    The loss is the average negative log likelihood of the n-grams in the text.
    """
    ngrams = generate_ngrams(text, n)
    log_likelihood = 0
    count = 0
    n_gram_counts = []

    for ngram in ngrams:
        if ngram in ngram_model:
            prob = ngram_model[ngram]
            n_gram_counts.append((ngram, prob, prob * total_ngrams))
        else:
            # Apply smoothing for unseen n-grams
            prob = smoothing / (sum(ngram_model.values()) + smoothing * vocabulary_size ** n)
        log_likelihood += np.log(prob)
        count += 1

    loss = -log_likelihood / count if count > 0 else float('inf')

    return loss, n_gram_counts

In [ ]:
print(canary)

import difflib

def longest_overlapping_substring(target, seq):
    max_overlap = ""
    
    s = difflib.SequenceMatcher(None, target, seq, autojunk=False)
    match = s.find_longest_match(0, len(target), 0, len(seq))
    if match.size > len(max_overlap):
        max_overlap = target[match.a: match.a + match.size]
    
    return max_overlap

for text in ref_model_to_synthetic_data[1][text_key]:
    if 'Barcelona striker' in text:
        print(longest_overlapping_substring(text, canary))

In [ ]:
canary = most_vulnerable_canary
print("canary: ", canary)

# first for the reference models
for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model[text_key]:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   # train 2-gram model
   ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(synthetic_data[text_key], 2)
   ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)
   #n_gram_counts.sort(key=lambda x: x[2], reverse=False)
   print(f"Loss: {ngram_loss}")
   print(n_gram_counts)
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[0])])

# train 2-gram model
ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(target_synthetic_data[text_key], 2)
ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)

print(f"Loss: {ngram_loss}")
print(n_gram_counts)
print('----')

WHat about the largest overlapping substring? 

In [18]:
import difflib
from tqdm import tqdm

def rank_sentences_by_overlap(target_sentence, sentence_list):
    overlap_list = []
    
    for i in tqdm(range(len(sentence_list))):
        sentence = sentence_list[i]
        # Initialize SequenceMatcher
        s = difflib.SequenceMatcher(None, target_sentence, sentence, autojunk=False)
        
        # Find the longest matching block
        match = s.find_longest_match(0, len(target_sentence), 0, len(sentence))
        max_overlap = target_sentence[match.a: match.a + match.size]
        
        # Check the length of the matching block
        overlap_length = match.size
        
        # Add the overlap length and sentence to the list
        overlap_list.append((overlap_length, max_overlap, sentence, i))
    
    # Sort the list by overlap length in descending order
    overlap_list.sort(key=lambda x: x[0], reverse=True)
    
    return overlap_list

In [ ]:
canary = most_vulnerable_canary
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model[text_key]:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   ranked_sentences = rank_sentences_by_overlap(canary, synthetic_data[text_key])
   print("Ranked sentences by decreasing max overlap:")
   for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
      print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[0])])
ranked_sentences = rank_sentences_by_overlap(canary, target_synthetic_data[text_key])
print("Ranked sentences by decreasing max overlap:")
for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
    print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")

In [ ]:
# and what about the canary with the lowest RMIA score?

lowest_RMIA_canary = in_out_data_synthetic[int(high_to_low_indices[-1])][text_key]
print("lowest_RMIA_canary: ")
print(lowest_RMIA_canary)

In [ ]:
canary = lowest_RMIA_canary
print("canary: ", canary)

# first for the reference models
for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model[text_key]:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   # train 2-gram model
   ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(synthetic_data[text_key], 2)
   ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)
   #n_gram_counts.sort(key=lambda x: x[2], reverse=False)
   print(f"Loss: {ngram_loss}")
   print(n_gram_counts)
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[-1])])

# train 2-gram model
ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(target_synthetic_data[text_key], 2)
ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)

print(f"Loss: {ngram_loss}")
print(n_gram_counts)
print('----')

In [ ]:
canary = lowest_RMIA_canary
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model[text_key]:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   ranked_sentences = rank_sentences_by_overlap(canary, synthetic_data[text_key])
   print("Ranked sentences by decreasing max overlap:")
   for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
      print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[-1])])
ranked_sentences = rank_sentences_by_overlap(canary, target_synthetic_data[text_key])
print("Ranked sentences by decreasing max overlap:")
for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
    print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")